In [ ]:
import numpy as np

def show_quant_levels(bit_width=4, symmetric=True):
    # 用 ASCII 画出量化级别在 [-1, 1] 上的分布
    Qmax = 2**bit_width-1; Qp = 2**(bit_width-1)-1
    if symmetric:
        lvs = np.linspace(-Qp, Qp, 2*Qp+1)/Qp
        lbl = f'Int{bit_width} symmetric ({2*Qp+1} levels)'
    else:
        lvs = np.linspace(0, Qmax, Qmax+1)/Qmax*2-1
        lbl = f'Uint{bit_width} asymmetric ({Qmax+1} levels)'
    bar = [' ']*61
    for v in lvs:
        pos = max(0, min(60, int(round((v+1)/2*60))))
        bar[pos] = '|'
    print(f'{lbl}:')
    print('  '+''.join(bar))
    print('  '+'-'*61)
    print('  -1'+' '*55+'1')


In [ ]:
def quantized_dot(x, w):
    # W4A16 点积: 权重量化为 Int4, 激活保持 FP16
    # 计算: 先用整数做点积, 最后乘 scale
    Qp = 2**(4-1)-1; Qn = -(2**(4-1))
    scale = np.max(np.abs(w)) / Qp
    qw = np.clip(np.round(w/scale), Qn, Qp).astype(np.int8)
    int_prod = x * qw
    int_sum = np.sum(int_prod)
    return np.dot(x, w), float(int_sum * scale), qw, scale

def quantized_matmul(X, W):
    # W4A16 matmul: per-channel 量化, broadcast scale
    oc, ic = W.shape
    Qp = 2**(4-1)-1; Qn = -(2**(4-1))
    scale = np.max(np.abs(W), axis=1, keepdims=True) / Qp
    qW = np.clip(np.round(W / scale), Qn, Qp).astype(np.int8)
    Y_int = qW @ X
    return Y_int * scale


In [ ]:
if __name__ == '__main__':
    np.random.seed(42)

    print('1. Quantization level visualization')
    show_quant_levels(4, True)
    show_quant_levels(4, False)

    print('2. W4A16 dot product (step by step)')
    x = np.random.randn(6); w = np.random.randn(6)
    f32, qres, qw, s = quantized_dot(x, w)
    print(f'  weights (raw):  {[round(v,3) for v in w]}')
    print(f'  weights (Int4): {list(qw)}')
    print(f'  scale:          {s:.4f}')
    print(f'  dot (FP32):     {f32:.4f}  dot (W4A16): {qres:.4f}  error: {abs(f32-qres):.6f}')

    print('\n3. Batch matmul simulation')
    W = np.random.randn(3, 6) * 0.5; X = np.random.randn(6, 4)
    Y_f32 = W @ X
    Y_q = quantized_matmul(X, W)
    mse = np.mean((Y_f32 - Y_q)**2)
    sqnr = 10*np.log10(np.mean(Y_f32**2)/(mse+1e-12))
    print(f'  W[{W.shape[0]}x{W.shape[1]}], X[{X.shape[0]}x{X.shape[1]}]')
    print(f'  matmul SQNR: {sqnr:.1f} dB')
    print(f'  max error: {np.max(np.abs(Y_f32-Y_q)):.4f}')
